In [1]:
import json
import pandas as pd
from Bio import AlignIO
import matplotlib.pyplot as plt
import numpy as np

# ==== FILES ====
gene = "SEC16B_ENSGALT00000066144.2"
prefix="trimmed_prank_"

PATH = "/lustre/fs5/jarv_lab/scratch/adenisova/Inno_2025/tmp"
json_file = f"{PATH}/{prefix}meme_prank_small_results_inno/{gene}.json"
out_file = f"{PATH}/{prefix}meme_prank_small_results_inno/{gene}.out"

# SLC16A6_rna-XM_015279952.2.prot.fa
# aln_file = f"{PATH}/{gene}.prot.fa"
aln_file  = f"{PATH}/trimmed_prank_small_prank_fa_proteins/{gene}.fasta"

alignment = AlignIO.read(aln_file, "fasta")
aln_len = alignment.get_alignment_length()

foreground_species = pd.read_csv(
    "/lustre/fs5/jarv_lab/scratch/adenisova/Inno_2025/initial_data/foreground_species.tsv",
    sep="\s+",
)
df = pd.read_csv(f"{PATH}/{prefix}absrel_e_summary_per_gene_filter.csv", sep=",")
branches = df[df["gene"] == gene]["sig_branches"].values[0]
print(branches)

print(f"aln_len: {aln_len}")

['Agelaius_phoeniceus', 'Egretta_garzetta', 'Falco_peregrinus', 'Falco_tinnunculus', 'Parus_major', 'Passer_domesticus', 'Sturnus_vulgaris']
aln_len: 968


In [2]:
from Bio import AlignIO
import pandas as pd

# Читаем таблицу
df = foreground_species

# Сортируем: сначала по Pair_Number, затем high > low, затем по алфавиту для low
df['inno_rank'] = df['Inno'].apply(lambda x: 0 if x == 'high' else 1)
df = df.sort_values(['Pair_Number', 'inno_rank', 'Scientific_Name'])

# Читаем выравнивание
alignment = AlignIO.read(aln_file, 'fasta')

# Создаем словарь для быстрого доступа
seq_dict = {record.id: record for record in alignment}

# Сохраняем в нужном порядке
with open(f"{PATH}/filtered_alignments/{gene}.fasta", 'w') as f:
    for _, row in df.iterrows():
        species = row['Scientific_Name']
        if species in seq_dict:
            record = seq_dict[species]
            # print(record.description)
            if record.description in branches:
                record.description += "_selected"
            f.write(f">{record.description}\n{str(record.seq)}\n")

In [3]:
with open(out_file) as f:
    file = f.read()

lines_of_table = []
is_Table = False
header = None
for line in file.split("\n"):
    if is_Table and line.strip() != "" and not line.startswith("###"):
        if not header:
            header = [i.strip() for i in line.split("|") if i != ""]
        else:
            line_comp = [i.strip() for i in line.split("|") if i != ""]
            line_final =  line_comp[:7]
            line_final.append(','.join(line_comp[7:10]))
            lines_of_table.append(line_final)
            

    if line.startswith("### For partition 1 these sites are significant at p <=0.1"):
        is_Table = True
    
lines_of_table = lines_of_table[1:]
df_great = pd.DataFrame(lines_of_table, columns=header)
df_great

,Codon,Partition,alpha,"non-syn rate (beta) distribution, rates : weights",LRT,Episodic selection detected?,# branches,List of most common codon substitutions at this site
0,78,1,0.000,0.00/65.30 : 0.01/0.99,5.765,"Yes, p = 0.0254",1,"[2]caG>caC,[1]cCc>cAc,cCC>cAG,Ccc>Tcc"
1,99,1,0.000,0.00/53.93 : 0.01/0.99,5.689,"Yes, p = 0.0264",1,"[3]caC>caG,[2]cAc>cGc,[1]CAc>ACc,caC>caA,Cac>Tac"
2,114,1,0.000,0.00/43.73 : 0.01/0.99,5.379,"Yes, p = 0.0310",1,"[2]aGc>aAc,Agc>Ggc,[1]aGc>aCc"
3,163,1,1.832,0.95/467.02 : 0.63/0.37,4.207,"Yes, p = 0.0568",1,"[1]Gga>Cga,gGa>gCa,ggA>ggC,ggC>ggT"
4,167,1,1.575,0.80/242.99 : 0.41/0.59,8.814,"Yes, p = 0.0054",2,"[10]Ggc>Agc,[1]gGa>gCa,GGc>ACc,gGc>gAc,ggC>ggA..."
5,204,1,1.632,0.84/79.88 : 0.43/0.57,3.294,"Yes, p = 0.0915",0,"[7]Gtc>Atc,[2]aTc>aCc,[1]aCc>aTc,Acc>Gcc,atC>a..."
6,235,1,0.000,0.00/35.88 : 0.01/0.99,6.256,"Yes, p = 0.0198",1,[1]Ccc>Tcc
7,260,1,0.960,0.00/31.70 : 0.01/0.99,4.023,"Yes, p = 0.0625",0,"[7]Ggt>Agt,[1]gGt>gAt,gGt>gCt,ggT>ggC"
8,377,1,1.159,0.00/27.47 : 0.01/0.99,3.662,"Yes, p = 0.0754",1,"[1]aGa>aAa,aGa>aCa,agA>agG,Aga>Gga,aGg>aAg"
9,425,1,0.000,0.00/26.04 : 0.01/0.99,4.277,"Yes, p = 0.0548",0,"[5]aAc>aGc,[3]aaC>aaA,[2]aAc>aCc"


In [5]:
df_great["# branches"] = df_great["# branches"].apply(lambda x: int(x)) 

one_branch = df_great[df_great["# branches"] == 1]["Codon"].to_list()

two_branch = df_great[df_great["# branches"] == 2]["Codon"].to_list()

three_branch = df_great[df_great["# branches"] == 3]["Codon"].to_list()

couple_branch = df_great[df_great["# branches"] > 3]["Codon"].to_list()

In [10]:
from pymsaviz import MsaViz, get_msa_testdata
# trimmed_small_fa_proteins filtered_alignments

for i in range(aln_len//100 + 1):
    start = 1+i*200
    end = (i+1)*200
    
    if start > aln_len:
        continue
    if end > aln_len:
        end = aln_len

    mv = MsaViz(
        f"{PATH}/filtered_alignments/{gene}.fasta",
        # aln_file, 
        show_count=True,
        show_consensus=True,
        wrap_length=100,
        start=start,
        end=end,
        color_scheme="Identity",
    )

    mv.add_markers([int(i) for i in one_branch], color = "blue")
    mv.add_markers([int(i) for i in two_branch], color = "green")
    mv.add_markers([int(i) for i in three_branch], color = "yellow")
    mv.add_markers([int(i) for i in couple_branch], color = "red")
    # mv.add_markers([int(i) for i in df_great["Codon"].tolist()])
    mv.savefig(f"{PATH}/alignment_pics_final/{gene}_{1+i*100}_{101+i*100}.pdf", dpi=300)

In [44]:
absrel = pd.read_csv(f"{PATH}/trimmed_prank_absrel_e_summary_per_gene_filter.csv", sep = ",")
absrel

# trimmed_prank_meme_prank_small_results_inno/rna-XM_015293919.2.out

def get_table_from_out(out_file):
    with open(out_file) as f:
        file = f.read()

    lines_of_table = []
    is_Table = False
    header = None
    for line in file.split("\n"):
        if is_Table and line.strip() != "" and not line.startswith("###"):
            if not header:
                header = [i.strip() for i in line.split("|") if i != ""]
            else:
                line_comp = [i.strip() for i in line.split("|") if i != ""]
                line_final =  line_comp[:7]
                line_final.append(','.join(line_comp[7:10]))
                lines_of_table.append(line_final)
                

        if line.startswith("### For partition 1 these sites are significant at p <=0.1"):
            is_Table = True
        
    lines_of_table = lines_of_table[1:]
    df_great = pd.DataFrame(lines_of_table, columns=header)
    return df_great

tables = []
for gene in absrel["gene"]:
    table = get_table_from_out(f"{PATH}/trimmed_prank_meme_prank_small_results_inno/{gene}.out")
    table["gene"] = gene

    tables.append(table)

In [50]:
table_final = pd.concat(tables)
table_final = table_final.dropna()#

table_final["# branches"].min()

'0'